### Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")

print("Libraries imported successfully.")

### Clean Up Data

<small>Read the data and convert into dataframe</small>

In [ ]:
# read data from csv
df = pd.read_csv('dirty_cafe_sales.csv')
print("Data loaded successfully.")

<small>Check the data set type using info</small>

In [ ]:
df.info()

<small>Check how the actual values look</small>

In [ ]:
df.head(20)

<small>Check unique values to sort out Erroneous data types</small>

In [ ]:
# check column unique values 
df.nunique()

# get for the Item, Price Per Unit, Payment Method, Location
unique_items = df['Item'].unique()
unique_prices = df['Price Per Unit'].unique().tolist()
unique_payment_methods = df['Payment Method'].unique().tolist()
unique_locations = df['Location'].unique().tolist()


print("Unique Items:", unique_items)
print("Unique Prices:", unique_prices)
print("Unique Payment Methods:", unique_payment_methods)   
print("Unique Locations:", unique_locations)

# take note of erroneous data types
error_values = ["ERROR", "UNKNOWN", "NAN"]

<small>Check for null values per column</small>

In [ ]:
print("Null Values in Each Column:")
print(df.isnull().sum())

<small>Check the data types</small>

In [ ]:
df.info()

<small>Convert to appropriate data type, apply coerce on errors to change to NaN or NaT</small>

In [ ]:
# convert to correct data types

# numerical columns
df['Quantity'] = pd.to_numeric(df['Quantity'], errors='coerce')
df['Price Per Unit'] = pd.to_numeric(df['Price Per Unit'], errors='coerce')

df['Total Spent'] = pd.to_numeric(df['Total Spent'], errors='coerce')
df['Total Price'] = df['Quantity'] * df['Price Per Unit']

#categorical columns
df['Item'] = df['Item'].astype('string')
df['Payment Method'] = df['Payment Method'].astype('string')
df['Location'] = df['Location'].astype('string')

#datetime column
df['Transaction Date'] = pd.to_datetime(df['Transaction Date'], errors='coerce')


<small>Recheck newly casted data types</small>

In [ ]:
df.info()

<small>Create a copy of the data frame. Make a helper function that converts values from the error_values list taken to a panda (pd.NA) or null value type in pandas. apply it on the copied dataframe, with axis set to "1" to apply to the rows</small>

In [ ]:

df_clean = df.copy()

def clean_data(row): 
    for col in df_clean.columns:
        if isinstance(row[col], str):
            if row[col].upper() in error_values:
                row[col] = pd.NA
    return row

df_clean = df_clean.apply(clean_data, axis=1)


<small>Check actual null values are reflected in rows</small>

In [ ]:
df_clean.head(20)

<small>Recheck the number of nulls in the columns to see that they increased</small>

In [ ]:
df_clean.isna().sum()

<small>Use different approaches to fill pd.NA types with appropriate values. Medians for numerical values, Mode for categorical. Total Price is a column calculated using other columns instead</small>

In [ ]:
# fill data based on type, median or mode

# Item - mode
df_clean['Item']= df_clean['Item'].fillna(df_clean['Item'].mode()[0], )

#Quantity - median
df_clean['Quantity']= df_clean['Quantity'].fillna(df_clean['Quantity'].median(), )

# Price Per Unit - median
df_clean['Price Per Unit']= df_clean['Price Per Unit'].fillna(df_clean['Price Per Unit'].median(), )

# Total Spent - recalculate
df_clean['Total Spent'] = df_clean['Total Spent'].fillna(df_clean['Total Spent'].median())

# Payment Method - mode
df_clean['Payment Method']= df_clean['Payment Method'].fillna(df_clean['Payment Method'].mode()[0],)

# Location - mode
df_clean['Location']= df_clean['Location'].fillna(df_clean['Location'].mode()[0], )

# Transaction Date - past unreachable date
default_time = pd.Timestamp("1990-01-01 00:00:00")
df_clean['Transaction Date']= df_clean['Transaction Date'].fillna(default_time)

#Total Price - recalculate
df_clean['Total Price'] = df_clean['Quantity'] * df_clean['Price Per Unit']

print("Succesfully turned null values in non-null.")

<small>Recheck and confirm that there are no null values present in each column </small>

In [ ]:
df_clean.isna().sum()

### Feature Engineering

<small>Add a feature to categorize an item to either a 'food' or 'drink' depending on the unique item value found in the item column </small>

In [ ]:
clean_unique_items = df_clean['Item'].unique().tolist()

<small>Nased on the data of the unique values, categorize them into food and drink</small>

In [ ]:
drinks = ['coffee', 'smoothie', 'juice', 'tea']
food = ['cake', 'cookie', 'salad', 'sandwich']

<small>Make a new column that lists the item as either a food or drink depending on the defined lists above</small>

In [ ]:
df_clean["Item Type"] = df_clean["Item"].apply(
    lambda x: "Frink" if pd.notna(x) and x.lower() in drinks else "Food"
)

<small>Check if the new column is added</small>

In [ ]:
df_clean.head()

<small>Add more columns to prepare for analysis</small>

In [ ]:
# transaction day
df_clean['Transaction Day'] = df_clean['Transaction Date'].dt.day_name()

# transaction month 
df_clean['Transaction Month'] = df_clean['Transaction Date'].dt.month_name()

# transaction year
df_clean['Transaction Year'] = df_clean['Transaction Date'].dt.year

print('Successfully extracted added datetime features.')


### Analysis

<small>Total Sold Per Item, and total earnings</small>

In [ ]:
total_sold_per_item = df_clean.groupby('Item')[['Quantity', 'Total Price']].sum().reset_index().sort_values(by='Quantity', ascending=False)
total_sold_per_item

<small>Most to least sold item per day</small>

In [ ]:
popular_items_per_day = df_clean.groupby(['Transaction Day', 'Item'])[['Quantity', 'Total Price']].sum().reset_index()
popular_items_per_day = popular_items_per_day.sort_values(['Transaction Day', 'Quantity'], ascending=[True, False])
popular_items_per_day

<small>Most to least sold item per month </small>

In [ ]:
popular_items_per_month = df_clean.groupby(['Transaction Month', 'Item'])[['Quantity', 'Total Price']].sum().reset_index()
popular_items_per_month = popular_items_per_month.sort_values(['Transaction Month', 'Quantity'], ascending=[True, False])

popular_items_per_month

<small>Most to least sold item per year</small>

In [ ]:
popular_items_per_year = df_clean.groupby(['Transaction Year', 'Item'])[['Quantity', 'Total Price']].sum().reset_index()
popular_items_per_year = popular_items_per_year.sort_values(['Transaction Year', 'Quantity'], ascending=[True, False])
popular_items_per_year

<small>Most Popular Location Types</small>

In [ ]:
most_popular_location = df_clean.groupby('Location')[['Quantity', 'Total Price']].sum().reset_index().sort_values(by='Quantity', ascending=False)
most_popular_location

<small>Most Popular Payment Methods</small>

In [ ]:
most_popular_payment_method = df_clean.groupby('Payment Method')[['Quantity', 'Total Price']].sum().reset_index().sort_values(by='Quantity', ascending=False)
most_popular_payment_method

### Visualization

<small>Total Sold Per Item, and total earnings (Visualization)</small>

In [ ]:

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Define manual colors
color_quantity = "skyblue"
color_price = "salmon"

# Barplot for total quantity sold
sns.barplot(
    data=total_sold_per_item,
    x="Item", y="Quantity",
    ax=axes[0], color=color_quantity
)
axes[0].set_title("Total Sold Per Item")
axes[0].set_xlabel("Item")
axes[0].set_ylabel("Quantity")
axes[0].tick_params(axis="x", rotation=45)

# Barplot for total earnings
sns.barplot(
    data=total_sold_per_item,
    x="Item", y="Total Price",
    ax=axes[1], color=color_price
)
axes[1].set_title("Total Earnings Per Item")
axes[1].set_xlabel("Item")
axes[1].set_ylabel("Total Price")
axes[1].tick_params(axis="x", rotation=45)

plt.tight_layout()
plt.show()


<small>Most to least sold item per day(Visualization)</small>

In [ ]:
pivot = popular_items_per_day.pivot(index="Item", columns="Transaction Day", values="Quantity")
pivot = pivot.astype(int)   

plt.figure(figsize=(10,6))
sns.heatmap(pivot, annot=True, fmt="d", cmap="Blues")
plt.title("Most to Least Sold Items per Day (Quantity)")
plt.ylabel("Item")
plt.xlabel("Transaction Day")
plt.show()


<small>Most to least sold item per month(Visualization) </small>

In [ ]:

pivot_month = popular_items_per_month.pivot(
    index="Item", 
    columns="Transaction Month", 
    values="Quantity"
)


pivot_month = pivot_month.fillna(0).astype(int)

plt.figure(figsize=(12, 7))
sns.heatmap(
    pivot_month, 
    annot=True, fmt="d", cmap="Blues"
)
plt.title("Most to Least Sold Items per Month (Quantity)")
plt.ylabel("Item")
plt.xlabel("Transaction Month")
plt.show()


<small>Most to least sold item per year(Visualization) </small>

In [ ]:

pivot_year = popular_items_per_year.pivot(
    index="Item",
    columns="Transaction Year",
    values="Quantity"
)

pivot_year = pivot_year.fillna(0).astype(int)

plt.figure(figsize=(12, 7))
sns.heatmap(
    pivot_year,
    annot=True, fmt="d", cmap="Blues"
)
plt.title("Most to Least Sold Items per Year (Quantity)")
plt.ylabel("Item")
plt.xlabel("Transaction Year")
plt.show()


<small>Most Popular Location Types(Visualization)</small>

In [ ]:
plt.figure(figsize=(8,8))
plt.pie(
    most_popular_location["Quantity"],
    labels=most_popular_location["Location"],
    autopct="%1.1f%%",
    startangle=140,
    colors=plt.cm.Paired.colors
)
plt.title("Share of Total Quantity Sold by Location")
plt.show()

# Pie chart for Total Price (earnings) per location
plt.figure(figsize=(8,8))
plt.pie(
    most_popular_location["Total Price"],
    labels=most_popular_location["Location"],
    autopct="%1.1f%%",
    startangle=140,
    colors=plt.cm.Set3.colors
)
plt.title("Share of Total Earnings by Location")
plt.show()


In [ ]:
plt.figure(figsize=(8,8))
plt.pie(
    most_popular_payment_method["Quantity"],
    labels=most_popular_payment_method["Payment Method"],
    autopct="%1.1f%%",
    startangle=140,
    colors=plt.cm.Paired.colors
)
plt.title("Share of Total Quantity Sold by Payment Method")
plt.show()

# Pie chart for Total Price (earnings) per payment method
plt.figure(figsize=(8,8))
plt.pie(
    most_popular_payment_method["Total Price"],
    labels=most_popular_payment_method["Payment Method"],
    autopct="%1.1f%%",
    startangle=140,
    colors=plt.cm.Set3.colors
)
plt.title("Share of Total Earnings by Payment Method")
plt.show()
